# 00 — контроль stepa330 и прогнозы для следующих экспериментов

Автономный Kaggle Notebook. Включите **GPU** и подключите dataset с
`train_solo_track.parquet` и `test_solo_track.parquet`. Репозиторий загружать не нужно.
Зависимости: numpy, pandas, pyarrow, catboost (обычно уже доступны в Kaggle).
Если import не проходит: `%pip install numpy pandas pyarrow catboost`, затем restart kernel.

Контроль: recent75, CatBoost depth=7 / learning_rate=0.05 / MAE, цель /100000.
h1 — CatBoost; h2–h8 — 70% CatBoost + 30% profile28; общая аддитивная поправка.
Настройки уже выбраны в проекте: здесь нет повторного поиска победителя.
Старые ноутбуки не изменяются. GPU-обучение не гарантирует побитового совпадения.

**Две отдельные проверки:** legacy повторяет прежние границы и early stopping;
weekly строит шесть недель с обновлением модели перед каждой неделей и фиксированными
1500 деревьями. Последующие недели не участвуют в fit или early stopping weekly.
Эти параметры уже выбирались на истории проекта: прогнозы причинны по fit,
но не являются новым слепым экспериментом по выбору параметров.

По умолчанию 64 fit: 8 legacy + 48 weekly + 8 full. Один GPU на fit;
`GPU_DEVICES='0:1'` можно задать для двух GPU, ускорение нужно измерить.
Не запускайте два экземпляра в одну RUN_ROOT.

**Продолжение:** в той же файловой системе просто Run All. После каждой модели
сохраняются checkpoint и results.zip; после горизонта — checkpoint.zip со всеми моделями.
После штатной ошибки обучения checkpoint.zip также пересобирается. При принудительном
завершении сессии доступен последний полностью сохранённый архив.
Для новой сессии сохраните checkpoint.zip, распакуйте его в `/kaggle/working`
(он содержит `baseline_00/`) и запустите с той же RUN_ROOT.
Альтернатива: подключите распакованный checkpoint как dataset и укажите RESTORE_DIR
на папку с manifest.json; RUN_ROOT должна ещё отсутствовать.
Версии библиотек, хеши данных и кода проверяются, несовместимый запуск не продолжится.
Незавершённый fit запускается заново; уже завершённые модели пропускаются.


In [1]:
from pathlib import Path
import gc
import hashlib
import importlib.metadata
import json
import os
import platform
import time
import zipfile

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from catboost import CatBoostRegressor
from catboost.utils import get_gpu_device_count
from IPython.display import display

DATA = Path('/kaggle/input/datasets/stepzak/rwb-ds')
RUN_ROOT = Path('/kaggle/working/baseline_00')
RESTORE_DIR = None
GPU_DEVICES = '0'
THREAD_COUNT = 4
RUN_FULL_REFIT = True
WEEKLY_BLOCKS = 6
ITERATIONS = 1500
TARGET_SCALE = 100_000
RECENT_TRAIN_FRACTION = 0.75
TRAIN_FILE = 'train_solo_track.parquet'
TEST_FILE = 'test_solo_track.parquet'
TARGET = 'target_1h'
STATUSES = [f'status_{i}' for i in range(1, 7)]
STEP = pd.Timedelta(minutes=30)
LAG_MAX_LOOKBACK_HOURS = 24
LAG_MIN_PAIRS = 336
LAG_MIN_CORRELATION = 0.05
PIPELINE_HASH = '128d7b6b6eb2bdcfd998a9e63c65d2510d8902fa7e71e677815a1964095381ee'

if not (DATA / TRAIN_FILE).is_file():
    matches = [p.parent for p in Path('/kaggle/input').rglob(TRAIN_FILE)
               if (p.parent / TEST_FILE).is_file()]
    if len(matches) != 1:
        raise ValueError(f'Укажите DATA вручную: найдено {matches}')
    DATA = matches[0]
if get_gpu_device_count() < 1:
    raise RuntimeError('Включите GPU в настройках Kaggle. CPU fallback отключён.')
print('GPU available:', get_gpu_device_count(), 'devices:', GPU_DEVICES)
if WEEKLY_BLOCKS != 6:
    raise ValueError('Этот протокол требует шесть недель: warmup / tuning / holdout по две недели')

if RESTORE_DIR is not None:
    import shutil
    source = Path(RESTORE_DIR)
    if RUN_ROOT.exists():
        raise ValueError('RESTORE_DIR применяется только к отсутствующей RUN_ROOT')
    if not (source / 'manifest.json').is_file():
        raise ValueError('RESTORE_DIR должен содержать manifest.json и папки результатов')
    shutil.copytree(source, RUN_ROOT)
for name in ['models', 'predictions', 'reports', 'lags', 'checkpoints']:
    (RUN_ROOT / name).mkdir(parents=True, exist_ok=True)

def file_hash(path):
    result = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            result.update(block)
    return result.hexdigest()

def save_json(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.partial')
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, default=str))
    temporary.replace(path)

manifest = dict(
    pipeline_hash=PIPELINE_HASH,
    input_hashes={name: file_hash(DATA / name) for name in [TRAIN_FILE, TEST_FILE]},
    versions={name: importlib.metadata.version(name) for name in ['numpy', 'pandas', 'pyarrow', 'catboost']},
    python=platform.python_version(), devices=GPU_DEVICES, thread_count=THREAD_COUNT,
    weekly_blocks=WEEKLY_BLOCKS, iterations=ITERATIONS, target_scale=TARGET_SCALE,
    recent_fraction=RECENT_TRAIN_FRACTION,
    model=dict(depth=7, learning_rate=0.05, loss_function='MAE', random_seed=42),
    lag=dict(max_hours=LAG_MAX_LOOKBACK_HOURS, min_pairs=LAG_MIN_PAIRS, min_corr=LAG_MIN_CORRELATION),
)
manifest_path = RUN_ROOT / 'manifest.json'
if manifest_path.exists() and json.loads(manifest_path.read_text()) != manifest:
    raise ValueError('Данные, код, параметры или версии изменились. Используйте новую RUN_ROOT.')
save_json(manifest_path, manifest)
RUN_ID = hashlib.sha256(json.dumps(manifest, sort_keys=True).encode()).hexdigest()[:16]
print('run_id:', RUN_ID, 'output:', RUN_ROOT)


GPU available: 2 devices: 0
run_id: bc6b14b04336d41f output: /kaggle/working/baseline_00


## Данные и границы
Полная панель проверяется до создания признаков.

In [2]:
history = pd.read_parquet(DATA / TRAIN_FILE, columns=["route_id", "timestamp", TARGET, *STATUSES]).sort_values(["timestamp", "route_id"])
test = pd.read_parquet(DATA / TEST_FILE, columns=["id", "route_id", "timestamp"])
times = pd.DatetimeIndex(history.timestamp.unique())
routes = np.sort(history.route_id.unique())
if not (history.notna().all().all() and test.notna().all().all()):
    raise ValueError("В данных есть пропуски")
if not (not history.duplicated(["route_id", "timestamp"]).any()):
    raise ValueError("Повторяется ключ маршрута и времени")
if not (times.equals(pd.date_range(times[0], times[-1], freq=STEP))):
    raise ValueError("Нарушена получасовая сетка")
if not (len(history) == len(times) * len(routes)):
    raise ValueError("Неполная панель маршрутов")
if not (test.id.is_unique and not test.duplicated(["route_id", "timestamp"]).any()):
    raise ValueError("Повторы идентификаторов или ключей теста")
end = times[-1]
if not ((end.hour, end.minute) == (10, 30)):
    raise ValueError("Последняя точка истории должна быть в 10:30")
expected = pd.MultiIndex.from_product([routes, [end + h*STEP for h in range(1, 9)]],
                                      names=["route_id", "timestamp"])
if not (expected.difference(pd.MultiIndex.from_frame(test[["route_id", "timestamp"]])).empty):
    raise ValueError("Тест не покрывает все ожидаемые маршруты и горизонты")
if not (len(test) == len(expected)):
    raise ValueError("Число строк теста отличается от ожидаемого")
values = {name: history[name].to_numpy(dtype=np.float32).reshape(len(times), len(routes))
          for name in [TARGET, *STATUSES]}
del history
if not all(np.isfinite(array).all() for array in values.values()):
    raise ValueError("В числовых рядах есть бесконечные значения")

# Границы определяются до fit любых статистик, использующих target.
dates = times[(times.hour == 10) & (times.minute == 30) & (times + 8 * STEP <= end)]
if len(dates) <= 42:
    raise ValueError("Недостаточно дат для временного разбиения")
tune_start, holdout_start = dates[-28], dates[-14]
LAG_VALIDATION_FIT_BEFORE = tune_start.normalize()

## Исторические признаки
Формулы взяты из stepa330; pilot-признаки не используются контрольной моделью.

In [3]:
import pyarrow as pa
import pyarrow.parquet as pq

target = values[TARGET]

def past(array, positions):
    positions = np.asarray(positions, dtype=np.int64)
    valid = (positions >= 0) & (positions < len(array))
    result = array[np.clip(positions, 0, len(array) - 1)].astype(np.float32)
    result[~valid] = np.nan
    return result

def average_available(matrices):
    total = np.zeros_like(matrices[0], dtype=np.float32)
    count = np.zeros_like(total)
    for matrix in matrices:
        valid = np.isfinite(matrix)
        total += np.where(valid, matrix, 0)
        count += valid
    return np.divide(total, count, out=np.full_like(total, np.nan), where=count > 0)

def rolling(array, steps):
    return pd.DataFrame(array).rolling(steps, min_periods=1).mean().to_numpy(dtype=np.float32)

means = {(name, steps): rolling(array, steps)
         for name, array in values.items() for steps in (4, 12)}
level24 = rolling(target, 48)
positions = np.arange(len(times))
extra_signals = {}
for name in STATUSES[:3]:
    extra_signals[name + "_hourly"] = values[name] + past(values[name], positions - 1)
extra_signals["warehouse_total_load"] = sum(extra_signals[name + "_hourly"] for name in STATUSES[:3])
extra_signals["warehouse_processing_rate"] = (
    extra_signals["status_3_hourly"] / (extra_signals["status_1_hourly"] + np.float32(1e-5)))
for name in STATUSES[3:]:
    extra_signals[name + "_diff_hourly"] = values[name] - past(values[name], positions - 2)
extra_signals["hub_buffer"] = values["status_5"] - values["status_4"]
extra_signals["hub_gate_bottleneck"] = (
    extra_signals["status_4_diff_hourly"] - extra_signals["status_6_diff_hourly"])
extra_signals["target_acceleration"] = past(target, positions - 1) - past(target, positions - 2)
hub_signal = extra_signals["status_6_diff_hourly"]

def common_features(origins):
    common = {}
    for k in (0, 1, 2, 4):
        common[f"target_back_{k}"] = past(target, origins - k)
    for name, array in values.items():
        for steps in (4, 12):
            common[f"{name}_mean_{steps}"] = means[name, steps][origins]
        if name in STATUSES:
            common[f"{name}_latest"] = array[origins]
            if name in STATUSES[:3]:
                common[f"{name}_change_1h"] = array[origins] - past(array, origins - 2)
    for name, array in extra_signals.items():
        common[name] = array[origins]
    for lag in (48, 96):
        common[f"target_1h_lag_{lag}"] = past(target, origins - lag)
    common["warehouse_total_load_lag_48"] = past(extra_signals["warehouse_total_load"], origins - 48)
    common["status_6_diff_hourly_lag_48"] = past(hub_signal, origins - 48)
    common["target_vs_mean_6h"] = target[origins] - means[TARGET, 12][origins]
    recent = means[TARGET, 4][origins]
    usual = average_available([past(means[TARGET, 4], origins - day*48) for day in range(1, 8)])
    common["morning_level_ratio"] = np.divide(
        recent, usual, out=np.full_like(recent, np.nan), where=usual > 0)
    return common

CORRELATION_COLUMNS = ["selected_status6_lag_steps", "selected_status6_correlation",
                       "selected_status6_pair_count", "status6_lag_valid",
                       "lagged_status6_signal", "lagged_status6_signal_available"]
FEATURE_GROUPS = {
    "calendar": ["hour", "minute", "hour_float", "hour_sin", "hour_cos", "dayofweek",
                 "is_weekend", "weekday_sin", "weekday_cos"],
    "local_statuses": ["status_1_hourly", "status_2_hourly", "status_3_hourly",
                       "warehouse_total_load", "warehouse_processing_rate"],
    "hub_combinations": ["status_4_diff_hourly", "status_5_diff_hourly", "status_6_diff_hourly",
                         "hub_buffer", "hub_gate_bottleneck"],
    "historical_lags": ["target_1h_lag_48", "target_1h_lag_96",
                        "warehouse_total_load_lag_48", "status_6_diff_hourly_lag_48"],
    "target_change": ["target_acceleration"],
    "correlation_lag": CORRELATION_COLUMNS,
    "pilot_start_time": ["start_sin", "start_cos"],
    "pilot_peaks": ["peak7_mean_max", "peak7_mean_level",
                    "peak7_share_-1", "peak7_share_0", "peak7_share_1"],
}

## Корреляционные лаги
Отдельный словарь для каждой границы обучения.

In [4]:
def calculate_dynamic_shoulder(target_values, status6_diff, timestamps, route_ids, *,
                               fit_before, max_lookback_hours=24,
                               min_pairs=336, min_correlation=0.05, route_block_size=128):
    """Обучить статистический лаг маршрута на timestamp < fit_before; без print и мутации входов."""
    grid = pd.DatetimeIndex(timestamps)
    route_ids = np.asarray(route_ids)
    target_values, status6_diff = np.asarray(target_values), np.asarray(status6_diff)
    if target_values.shape != status6_diff.shape or target_values.shape != (len(grid), len(route_ids)):
        raise ValueError("Ожидаются два массива [время, маршрут] одной формы")
    if len(grid) == 0 or not grid.equals(pd.date_range(grid[0], periods=len(grid), freq=STEP)):
        raise ValueError("Для выбора лага нужна упорядоченная получасовая сетка без дубликатов")
    if len(np.unique(route_ids)) != len(route_ids):
        raise ValueError("Идентификаторы маршрутов повторяются")
    steps = float(max_lookback_hours) * 2
    if not np.isfinite(steps) or steps < 2 or not steps.is_integer():
        raise ValueError("Граница поиска должна задавать целое число получасов, не меньше двух")
    if not isinstance(min_pairs, (int, np.integer)) or min_pairs < 3:
        raise ValueError("min_pairs должен быть целым числом >=3")
    if not 0 <= min_correlation <= 1 or route_block_size < 1:
        raise ValueError("Некорректные параметры выбора корреляции")
    max_lag = int(steps)
    stop = int(grid.searchsorted(pd.Timestamp(fit_before), side="left"))
    # Первые две точки diff(2) неизвестны; общие якоря одинаковы для всех L.
    anchors = np.arange(max_lag + 2, stop)
    best_corr = np.full(len(route_ids), np.nan)
    best_lag = np.full(len(route_ids), np.nan)
    best_pairs = np.zeros(len(route_ids), dtype=np.int32)
    for start in range(0, len(route_ids), route_block_size):
        finish = min(start + route_block_size, len(route_ids))
        x = target_values[anchors, start:finish].astype(np.float64)
        for lag in range(2, max_lag + 1):
            z = status6_diff[anchors - lag, start:finish].astype(np.float64)
            valid = np.isfinite(x) & np.isfinite(z)
            count = valid.sum(axis=0)
            xx, zz = np.where(valid, x, 0), np.where(valid, z, 0)
            divisor = np.maximum(count, 1)
            sx, sz = xx.sum(axis=0), zz.sum(axis=0)
            covariance = (xx * zz).sum(axis=0) - sx * sz / divisor
            vx = np.maximum((xx * xx).sum(axis=0) - sx * sx / divisor, 0)
            vz = np.maximum((zz * zz).sum(axis=0) - sz * sz / divisor, 0)
            denominator = np.sqrt(vx * vz)
            corr = np.divide(covariance, denominator, out=np.full(finish-start, np.nan),
                             where=(count >= min_pairs) & (denominator > 0))
            corr = np.clip(corr, -1, 1)
            previous = best_corr[start:finish]
            better = np.isfinite(corr) & (~np.isfinite(previous) | (corr > previous))
            best_corr[start:finish][better] = corr[better]
            best_lag[start:finish][better] = lag
            best_pairs[start:finish][better] = count[better]
    accepted = np.isfinite(best_corr) & (best_corr > min_correlation)
    return pd.DataFrame({"route_id": route_ids,
                         "selected_status6_lag_steps": np.where(accepted, best_lag, np.nan),
                         "selected_status6_correlation": best_corr,
                         "selected_status6_pair_count": best_pairs,
                         "status6_lag_valid": accepted.astype(np.int8)})

def add_correlation_features(frame, h, lag_table, chunk_rows=200_000):
    """Применить готовый словарь; источник каждого значения <= собственного origin."""
    if h not in range(1, 9) or chunk_rows < 1:
        raise ValueError("Некорректный горизонт или размер блока")
    by_route = lag_table.set_index("route_id").reindex(routes)
    route_index = pd.Index(routes)
    lag = by_route.selected_status6_lag_steps.to_numpy(dtype=np.float32)
    quality = {name: by_route[name].to_numpy(dtype=np.float32) for name in CORRELATION_COLUMNS[:4]}
    for name in CORRELATION_COLUMNS:
        frame[name] = np.full(len(frame), np.nan, dtype=np.float32)
    column_positions = {name: frame.columns.get_loc(name) for name in CORRELATION_COLUMNS}
    for start in range(0, len(frame), chunk_rows):
        finish = min(start + chunk_rows, len(frame))
        block = frame.iloc[start:finish]
        origin = times.get_indexer(pd.DatetimeIndex(block.cutoff))
        route = route_index.get_indexer(block.route_id)
        if (origin < 0).any() or (route < 0).any():
            raise ValueError("Строка признаков не сопоставлена с историей")
        selected = lag[route]
        source = origin + h - np.nan_to_num(selected, nan=0).astype(np.int64)
        available = (np.isfinite(selected) & (selected >= h) & (source >= 0)
                     & (source <= origin) & (source < len(times)))
        signal = hub_signal[np.clip(source, 0, len(times)-1), route].astype(np.float32)
        available &= np.isfinite(signal)
        for name, array in quality.items():
            frame.iloc[start:finish, column_positions[name]] = array[route]
        frame.iloc[start:finish, column_positions["lagged_status6_signal"]] = np.where(available, signal, np.nan)
        frame.iloc[start:finish, column_positions["lagged_status6_signal_available"]] = available.astype(np.float32)
    return frame


## Признаки горизонта
Все источники значений доступны на момент cutoff.

In [5]:
def horizon_dataset(h, origins):
    slot = origins + h
    dates = times[origins] + h * STEP
    frame = pd.DataFrame({
        "cutoff": times[origins].repeat(len(routes)),
        "timestamp": dates.repeat(len(routes)),
        "route_id": np.tile(routes, len(origins)),
    })
    for name, matrix in common_features(origins).items():
        frame[name] = matrix.ravel()
    calendar = {
        "hour": dates.hour, "minute": dates.minute,
        "hour_float": dates.hour + dates.minute / 60.0,
        "dayofweek": dates.dayofweek, "is_weekend": (dates.dayofweek >= 5).astype(np.int8),
    }
    calendar["hour_sin"] = np.sin(2 * np.pi * calendar["hour_float"] / 24)
    calendar["hour_cos"] = np.cos(2 * np.pi * calendar["hour_float"] / 24)
    calendar["weekday_sin"] = np.sin(2 * np.pi * dates.dayofweek / 7)
    calendar["weekday_cos"] = np.cos(2 * np.pi * dates.dayofweek / 7)
    for name, array in calendar.items():
        frame[name] = np.repeat(array, len(routes))
    for name in CORRELATION_COLUMNS:
        frame[name] = np.float32(np.nan)
    for days in (1, 2, 7):
        frame[f"slot_days_back_{days}"] = past(target, slot - days*48).ravel()
    for name, days in [("slot_daily7", range(1, 8)), ("slot_weekly4", (7, 14, 21, 28))]:
        frame[name] = average_available([past(target, slot - d*48) for d in days]).ravel()
    profile = np.zeros((len(origins), len(routes)))
    counts = np.zeros_like(profile)
    for day in range(1, 29):
        level = past(level24, origins - day*48)
        adjusted = past(target, slot - day*48) * level24[origins] / np.maximum(level, 1)
        profile += np.nan_to_num(adjusted)
        counts += np.isfinite(adjusted)
    frame["profile28"] = np.divide(profile, counts, out=np.full_like(profile, np.nan), where=counts > 0).ravel()
    if h == 1:
        lower = np.zeros_like(target[origins])
        upper = past(target, origins - 2)
        for position in (origins - 1, origins):
            total = past(target, position)
            lower, upper = np.maximum(0, total - upper), np.minimum(total, total - lower)
        seasonal = frame[["slot_daily7", "slot_weekly4"]].mean(axis=1).to_numpy()
        frame["overlap"] = np.where((lower <= upper).ravel(),
                                    (lower + upper).ravel() / 2 + seasonal / 2, seasonal)
    else:
        frame["overlap"] = np.nan
    numeric = frame.columns.difference(["cutoff", "timestamp", "route_id"])
    return frame.astype({name: "float32" for name in numeric})


## История общего объёма
Агрегаты сохраняются для следующего эксперимента. При его обучении также нужны временные границы.

In [6]:
network_history = pd.DataFrame({'timestamp': times})
for name, array in values.items():
    network_history[name] = array.sum(axis=1, dtype=np.float64)
network_history.to_parquet(RUN_ROOT / 'network_history.parquet', index=False)
del network_history


## Протокол и сохранение
Legacy: train до 4 октября, tuning 4–17 октября, holdout 18–31 октября (для текущих данных).
Weekly: 20 сентября – 3 октября — warmup калибровки; 4–17 октября — tuning;
18–31 октября — ранее просмотренный holdout. Даты вычисляются из истории.
Weekly-контроль использует одну поправку по warmup, зафиксированную до tuning.
Окна 7/14/28 дней будут сравниваться в следующем ноутбуке.
Корреляционные словари fit только по меткам до границы. Train использует все
получасовые старты последних 75% доступных cutoffs отдельно для каждого горизонта.
Legacy tuning участвует в early stopping: его нельзя использовать как независимый OOF.
Показываем in-sample train и query метрики; query legacy объединяет tuning и holdout,
раздельная итоговая оценка появится в последней ячейке.


In [7]:
def metrics(frame, prediction):
    y = frame.y.to_numpy(dtype=np.float64)
    p = np.asarray(prediction, dtype=np.float64)
    if p.shape != y.shape or not np.isfinite(p).all() or not np.isfinite(y).all():
        raise ValueError('Некорректные факты или прогнозы')
    e = p - y
    daily = pd.DataFrame(dict(day=frame.cutoff.dt.normalize().to_numpy(), y=y,
                              absolute=np.abs(e), signed=e)).groupby('day').sum()
    if (daily.y <= 0).any():
        raise ValueError('Неопределённая метрика: нулевой объём дня')
    return dict(rows=len(frame), days=len(daily), WAPE=float(daily.absolute.sum()/daily.y.sum()),
                signed_bias=float(daily.signed.sum()/daily.y.sum()),
                score=float((daily.absolute.sum()+abs(daily.signed.sum()))/daily.y.sum()),
                daily_score=float(((daily.absolute+daily.signed.abs())/daily.y).mean()))

def atomic_parquet(frame, path):
    temporary = path.with_suffix('.partial.parquet')
    frame.to_parquet(temporary, index=False)
    temporary.replace(path)

def archive(name, include_models=False):
    destination = RUN_ROOT.parent / name
    temporary = destination.with_suffix('.partial.zip')
    with zipfile.ZipFile(temporary, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=1) as z:
        for path in sorted(RUN_ROOT.rglob('*')):
            if not path.is_file() or '.partial' in path.name:
                continue
            if path.suffix == '.cbm' and not include_models:
                continue
            z.write(path, str(Path(RUN_ROOT.name) / path.relative_to(RUN_ROOT)))
    temporary.replace(destination)
    print('Saved:', destination.name, f'{destination.stat().st_size / 2**20:.1f} MiB')

def completed(stage, h):
    marker = RUN_ROOT / 'checkpoints' / f'{stage}_h{h}.json'
    if not marker.exists():
        return None
    info = json.loads(marker.read_text())
    if info['run_id'] != RUN_ID:
        raise ValueError('Checkpoint принадлежит другому запуску')
    for rel, digest in info['files'].items():
        path = RUN_ROOT / rel
        if not path.is_file() or file_hash(path) != digest:
            raise ValueError(f'Повреждён checkpoint: {path}. Восстановите архив или используйте новую RUN_ROOT.')
    return info

def model_columns(frame):
    return [c for c in frame if c not in ['timestamp', 'cutoff', 'y', 'profile28', 'overlap']]

def recent_mask(frame, boundary):
    eligible = frame.timestamp < boundary
    cutoffs = pd.DatetimeIndex(frame.loc[eligible, 'cutoff'].unique()).sort_values()
    if len(cutoffs) == 0:
        raise ValueError('Пустой train')
    start = cutoffs[int(np.floor((1-RECENT_TRAIN_FRACTION)*len(cutoffs)))]
    return eligible & frame.cutoff.ge(start), start

def make_model(trees):
    return CatBoostRegressor(iterations=trees, depth=7, learning_rate=0.05,
                             loss_function='MAE', eval_metric='MAE', task_type='GPU',
                             devices=GPU_DEVICES, thread_count=THREAD_COUNT,
                             cat_features=['route_id'], random_seed=42,
                             allow_writing_files=False, verbose=300)

weekly_days = dates[-7*WEEKLY_BLOCKS:]
if len(weekly_days) != 42 or not weekly_days.equals(pd.date_range(weekly_days[0], periods=42, freq='D')):
    raise ValueError('Нужны 42 последовательные даты')
STAGES = [('legacy', tune_start.normalize(), dates[-28:])]
for number in range(WEEKLY_BLOCKS):
    block = weekly_days[number*7:(number+1)*7]
    STAGES.append((f'week{number+1}', block[0].normalize(), block))
if RUN_FULL_REFIT:
    STAGES.append(('full', end + STEP, pd.DatetimeIndex([end])))
stage_table = pd.DataFrame([dict(stage=s, fit_before=b, first=d.min(), last=d.max(), days=len(d))
                            for s,b,d in STAGES])
stage_table.to_csv(RUN_ROOT / 'reports' / 'stages.csv', index=False)
display(stage_table)
print('Всего fit:', len(STAGES)*8, '; weekly: фиксированные 1500 деревьев, без early stopping на будущей неделе')

lag_tables = {}
for stage, boundary, days in STAGES:
    lag_tables[stage] = calculate_dynamic_shoulder(
        target, hub_signal, times, routes, fit_before=boundary,
        max_lookback_hours=LAG_MAX_LOOKBACK_HOURS,
        min_pairs=LAG_MIN_PAIRS, min_correlation=LAG_MIN_CORRELATION)
    lag_tables[stage].to_csv(RUN_ROOT / 'lags' / f'{stage}.csv', index=False)

def build_history(h):
    path = RUN_ROOT / f'features_h{h}.partial.parquet'
    writer = None
    try:
        for start in range(0, len(times)-h, 48):
            origins = np.arange(start, min(start+48, len(times)-h))
            frame = horizon_dataset(h, origins)
            frame['y'] = target[origins+h].ravel()
            table = pa.Table.from_pandas(frame, preserve_index=False)
            if writer is None:
                writer = pq.ParquetWriter(path, table.schema, compression='zstd')
            writer.write_table(table)
    finally:
        if writer is not None:
            writer.close()
    frame = pd.read_parquet(path)
    path.unlink()
    return frame

def run_task(stage, boundary, days, h, data):
    old = completed(stage, h)
    if old is not None:
        print('Resume: skip', stage, h)
        return
    tic = time.perf_counter()
    data = add_correlation_features(data, h, lag_tables[stage])
    columns = model_columns(data)
    mask, start = recent_mask(data, boundary)
    if not (data.loc[mask, 'timestamp'] < boundary).all():
        raise ValueError('Train пересекает границу fit')
    if stage == 'full':
        query = horizon_dataset(h, np.array([len(times)-1]))
        query = add_correlation_features(query, h, lag_tables[stage])
        trees = completed('legacy', h)['trees']
    else:
        query = data.loc[data.cutoff.isin(days)].copy()
        trees = ITERATIONS
    if len(query) != len(days)*len(routes):
        raise ValueError('Неполная панель query')
    model = make_model(trees)
    x = data.loc[mask, columns]
    y = data.loc[mask, 'y'] / TARGET_SCALE
    if stage == 'legacy':
        tune = query.loc[query.cutoff < holdout_start]
        model.fit(x, y, eval_set=(tune[columns], tune.y/TARGET_SCALE),
                  early_stopping_rounds=100, use_best_model=True)
        info_through = tune.timestamp.max()
    else:
        model.fit(x, y)
        info_through = data.loc[mask, 'timestamp'].max()
    train_metrics = metrics(data.loc[mask], np.maximum(model.predict(x)*TARGET_SCALE, 0))
    del x, y
    prediction = np.maximum(model.predict(query[columns])*TARGET_SCALE, 0)
    keep = ['cutoff', 'timestamp', 'route_id', 'profile28', 'overlap']
    if stage != 'full':
        keep.append('y')
    result = query[keep].copy()
    result['horizon'] = h
    result['catboost'] = prediction
    result['seasonal'] = query[['slot_daily7', 'slot_weekly4']].mean(axis=1)
    result['raw_prediction'] = np.where(h == 1, prediction, .7*prediction+.3*query.profile28)
    result['stage'] = stage
    result['fit_before'] = boundary
    result['model_info_through'] = info_through
    result['run_id'] = RUN_ID
    result['selection_caveat'] = 'parameters_previously_selected_in_project'
    if stage == 'full':
        result = test.loc[test.timestamp.eq(end+h*STEP)].merge(
            result, on=['route_id', 'timestamp'], validate='one_to_one')
    pred_path = RUN_ROOT / 'predictions' / f'{stage}_h{h}.parquet'
    atomic_parquet(result, pred_path)
    model_path = RUN_ROOT / 'models' / f'{stage}_h{h}.cbm'
    temporary = model_path.with_suffix('.partial.cbm')
    model.save_model(str(temporary))
    temporary.replace(model_path)
    report = dict(stage=stage, horizon=h, run_id=RUN_ID, fit_before=str(boundary),
                  train_start=str(start), train_label_max=str(data.loc[mask, 'timestamp'].max()),
                  model_info_through=str(info_through), trees=model.tree_count_,
                  train=train_metrics, seconds=time.perf_counter()-tic, features=columns)
    if stage != 'full':
        report['query'] = metrics(result, prediction)
    importance_path = RUN_ROOT / 'reports' / f'{stage}_h{h}_importance.csv'
    pd.DataFrame(dict(feature=columns, importance=model.get_feature_importance())).to_csv(importance_path, index=False)
    report['files'] = {str(p.relative_to(RUN_ROOT)): file_hash(p) for p in [pred_path, model_path, importance_path]}
    save_json(RUN_ROOT / 'checkpoints' / f'{stage}_h{h}.json', report)
    print(stage, 'h=', h, 'trees=', model.tree_count_, 'seconds=', round(report['seconds']))
    display(pd.DataFrame([dict(split=k, **report[k]) for k in ['train', 'query'] if k in report]))
    del model, query, result
    gc.collect()
    archive('results.zip')

def run_all():
    for h in range(1, 9):
        if all(completed(s, h) is not None for s, _, _ in STAGES):
            print('Горизонт уже завершён:', h)
            continue
        print('Подготовка признаков h=', h)
        data = build_history(h)
        try:
            for stage, boundary, days in STAGES:
                run_task(stage, boundary, days, h, data)
        finally:
            del data
            gc.collect()
            archive('checkpoint.zip', include_models=True)


,stage,fit_before,first,last,days
0,legacy,2025-10-04 00:00:00,2025-10-04 10:30:00,2025-10-31 10:30:00,28
1,week1,2025-09-20 00:00:00,2025-09-20 10:30:00,2025-09-26 10:30:00,7
2,week2,2025-09-27 00:00:00,2025-09-27 10:30:00,2025-10-03 10:30:00,7
3,week3,2025-10-04 00:00:00,2025-10-04 10:30:00,2025-10-10 10:30:00,7
4,week4,2025-10-11 00:00:00,2025-10-11 10:30:00,2025-10-17 10:30:00,7
5,week5,2025-10-18 00:00:00,2025-10-18 10:30:00,2025-10-24 10:30:00,7
6,week6,2025-10-25 00:00:00,2025-10-25 10:30:00,2025-10-31 10:30:00,7
7,full,2025-11-01 11:00:00,2025-11-01 10:30:00,2025-11-01 10:30:00,1


Всего fit: 64 ; weekly: фиксированные 1500 деревьев, без early stopping на будущей неделе


## Обучение на GPU
Признаки обрабатываются по одному горизонту. Результаты каждой модели сохраняются атомарно.

In [8]:
run_all()


Подготовка признаков h= 1


Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 1.2783649	test: 1.4245204	best: 1.4245204 (0)	total: 452ms	remaining: 11m 18s
300:	learn: 0.7117429	test: 0.7795709	best: 0.7795709 (300)	total: 43.3s	remaining: 2m 52s
600:	learn: 0.7003171	test: 0.7662375	best: 0.7662375 (600)	total: 1m 26s	remaining: 2m 9s
900:	learn: 0.6942947	test: 0.7598438	best: 0.7598438 (900)	total: 2m 9s	remaining: 1m 26s
1200:	learn: 0.6901645	test: 0.7554052	best: 0.7554052 (1200)	total: 2m 52s	remaining: 43.1s
1499:	learn: 0.6871860	test: 0.7522575	best: 0.7522575 (1499)	total: 3m 35s	remaining: 0us
bestTest = 0.7522574637
bestIteration = 1499
legacy h= 1 trees= 1500 seconds= 230


,split,rows,days,WAPE,signed_bias,score,daily_score
0,train,2448000,52,0.257625,-0.034530,0.292154,0.291657
1,query,28000,28,0.253435,-0.048938,0.302373,0.302073


Saved: results.zip 1.4 MiB


Default metric period is 5 because MAE is/are not implemented for GPU


... (247 outputs left out)

Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 1.3137964	total: 243ms	remaining: 6m 4s
300:	learn: 0.9446892	total: 1m	remaining: 3m 59s
600:	learn: 0.9414009	total: 1m 58s	remaining: 2m 55s
900:	learn: 0.9399895	total: 2m 57s	remaining: 1m 57s
1200:	learn: 0.9390146	total: 3m 59s	remaining: 59s
1496:	learn: 0.9382358	total: 5m	remaining: 0us
full h= 8 trees= 1497 seconds= 318


,split,rows,days,WAPE,signed_bias,score,daily_score
0,train,3467000,73,0.3436,-0.039913,0.383513,0.383192


Saved: results.zip 19.6 MiB
Saved: checkpoint.zip 381.9 MiB


## Итоги и архивы
Основной критерий — средняя дневная WAPE + |Bias| по всем маршрутам и восьми горизонтам.
Отдельно сохраняются pooled score, signed bias, WAPE, дни и горизонты.
Для submission контрольная поправка переоценивается на 28 legacy-датах,
число деревьев берётся из legacy, все доступные метки участвуют в final recent75 refit.
Это воспроизведение текущего подхода, не автоматический выбор новой калибровки.

Сохраните выполненный `.ipynb`, **results.zip**, **models.zip**.
Для переноса незавершённого запуска сохраните **checkpoint.zip**.
В results.zip есть manifest, прогнозы, метрики, тайминги, лаги, агрегированная история
и контрольный submission. Полные матрицы признаков в архивы не входят.


In [9]:
def collect(stage_names):
    pieces = []
    for stage in stage_names:
        for h in range(1, 9):
            if completed(stage, h) is None:
                raise ValueError(f'Не завершён {stage}, h{h}')
            pieces.append(pd.read_parquet(RUN_ROOT / 'predictions' / f'{stage}_h{h}.parquet'))
    frame = pd.concat(pieces, ignore_index=True).sort_values(['cutoff', 'horizon', 'route_id']).reset_index(drop=True)
    if frame.duplicated(['cutoff', 'route_id', 'horizon']).any():
        raise ValueError('Дубли ключей прогнозов')
    if not np.isfinite(frame[['catboost', 'raw_prediction', 'profile28']].to_numpy()).all():
        raise ValueError('Неконечный прогноз')
    return frame

def finish():
    legacy = collect(['legacy'])
    tuning = legacy.cutoff < holdout_start
    offset = float((legacy.loc[tuning, 'y']-legacy.loc[tuning, 'raw_prediction']).mean())
    legacy['y_pred'] = np.maximum(legacy.raw_prediction + offset, 0)
    legacy['period'] = np.where(tuning, 'tuning', 'holdout')
    legacy['calibration_info_through'] = legacy.loc[tuning, 'timestamp'].max()
    atomic_parquet(legacy, RUN_ROOT / 'legacy_predictions.parquet')

    weekly = collect([f'week{i}' for i in range(1, WEEKLY_BLOCKS+1)])
    if not (weekly.model_info_through < weekly.cutoff).all():
        raise ValueError('Недельная модель использует будущее')
    warmup = weekly.cutoff < tune_start
    fixed_offset = float((weekly.loc[warmup, 'y']-weekly.loc[warmup, 'raw_prediction']).mean())
    weekly['y_pred'] = np.where(warmup, np.nan, np.maximum(weekly.raw_prediction+fixed_offset, 0))
    weekly['period'] = np.where(warmup, 'warmup', np.where(weekly.cutoff < holdout_start, 'tuning', 'holdout'))
    weekly['calibration_info_through'] = weekly.loc[warmup, 'timestamp'].max()
    atomic_parquet(weekly, RUN_ROOT / 'weekly_predictions.parquet')
    rows, daily_rows = [], []
    for protocol, frame in [('legacy', legacy), ('weekly', weekly.loc[~warmup])]:
        for period, part in frame.groupby('period'):
            for h in [0, *range(1, 9)]:
                block = part if h == 0 else part.loc[part.horizon.eq(h)]
                for col in ['catboost', 'raw_prediction', 'y_pred']:
                    rows.append(dict(protocol=protocol, period=period, horizon=h, prediction=col,
                                     **metrics(block, block[col])))
        for day, part in frame.groupby('cutoff'):
            for col in ['catboost', 'raw_prediction', 'y_pred']:
                daily_rows.append(dict(protocol=protocol, cutoff=day, saturday=day.dayofweek == 5,
                                       period=part.period.iloc[0], prediction=col, **metrics(part, part[col])))
    report = pd.DataFrame(rows)
    report.to_csv(RUN_ROOT / 'reports' / 'metrics.csv', index=False)
    pd.DataFrame(daily_rows).to_csv(RUN_ROOT / 'reports' / 'daily_metrics.csv', index=False)
    display(report.query('horizon == 0'))
    calibration = dict(legacy_tuning_offset=offset, weekly_warmup_offset=fixed_offset,
                       weights=dict(h1_catboost=1.0, h2_h8_catboost=0.7, h2_h8_profile28=0.3),
                       legacy_reference=dict(holdout_WAPE=0.321816, holdout_score=0.333315,
                                             holdout_daily_score=0.340499),
                       warning='Сравнение с сохранённым запуском, не битовая идентичность GPU. Параметры уже выбирались в проекте.')
    if RUN_FULL_REFIT:
        future = collect(['full'])
        final_offset = float((legacy.y-legacy.raw_prediction).mean())
        future['y_pred'] = np.maximum(future.raw_prediction+final_offset, 0)
        future['calibration_info_through'] = legacy.timestamp.max()
        atomic_parquet(future, RUN_ROOT / 'full_predictions.parquet')
        submission = test[['id']].merge(future[['id', 'y_pred']], on='id', how='left', validate='one_to_one')
        if (len(submission) != len(test) or not submission.id.equals(test.id.reset_index(drop=True))
                or not np.isfinite(submission.y_pred).all() or (submission.y_pred < 0).any()):
            raise ValueError('Некорректный submission')
        submission.to_csv(RUN_ROOT / 'submission.csv', index=False)
        calibration['submission_offset'] = final_offset
    save_json(RUN_ROOT / 'reports' / 'calibration.json', calibration)
    timings = [json.loads(path.read_text()) for path in sorted((RUN_ROOT / 'checkpoints').glob('*.json'))]
    pd.DataFrame([{k: r[k] for k in ['stage', 'horizon', 'trees', 'seconds', 'train_start', 'fit_before']}
                  for r in timings]).to_csv(RUN_ROOT / 'reports' / 'timings.csv', index=False)
    save_json(RUN_ROOT / 'status.json', dict(complete=True, run_id=RUN_ID, tasks=len(timings),
                                          full_refit=RUN_FULL_REFIT))
    archive('results.zip')
    archive('checkpoint.zip', include_models=True)
    destination = RUN_ROOT.parent / 'models.zip'
    temporary = destination.with_suffix('.partial.zip')
    with zipfile.ZipFile(temporary, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=1) as z:
        for folder in ['models', 'lags']:
            for path in sorted((RUN_ROOT / folder).glob('*')):
                if path.is_file() and '.partial' not in path.name:
                    z.write(path, str(Path(RUN_ROOT.name) / path.relative_to(RUN_ROOT)))
        z.write(RUN_ROOT / 'manifest.json', f'{RUN_ROOT.name}/manifest.json')
    temporary.replace(destination)
    print('Готово. Сохраните выполненный notebook, results.zip и models.zip.')
    print('Для продолжения в новой сессии нужен checkpoint.zip, а не results.zip.')

finish()


,protocol,period,horizon,prediction,rows,days,WAPE,signed_bias,score,daily_score
0,legacy,holdout,0,catboost,112000,14,0.322205,-5.343598e-02,0.375641,0.375369
1,legacy,holdout,0,raw_prediction,112000,14,0.320914,-3.515689e-02,0.356071,0.356788
2,legacy,holdout,0,y_pred,112000,14,0.321818,-1.149550e-02,0.333314,0.340513
27,legacy,tuning,0,catboost,112000,14,0.324598,-3.525890e-02,0.359857,0.359765
28,legacy,tuning,0,raw_prediction,112000,14,0.324075,-2.336006e-02,0.347435,0.348022
29,legacy,tuning,0,y_pred,112000,14,0.325446,5.902925e-18,0.325446,0.343048
54,weekly,holdout,0,catboost,112000,14,0.320547,-4.994371e-02,0.370491,0.370196
55,weekly,holdout,0,raw_prediction,112000,14,0.320093,-3.267367e-02,0.352766,0.353573
56,weekly,holdout,0,y_pred,112000,14,0.321790,9.538207e-04,0.322744,0.338366
81,weekly,tuning,0,catboost,112000,14,0.324299,-3.647713e-02,0.360777,0.360696


Saved: results.zip 41.3 MiB
Saved: checkpoint.zip 403.6 MiB
Готово. Сохраните выполненный notebook, results.zip и models.zip.
Для продолжения в новой сессии нужен checkpoint.zip, а не results.zip.
